In [1]:
!pip install transformers datasets accelerate torch -q

In [3]:
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

from datasets import Dataset

In [4]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

print("GPT-2 loaded successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 loaded successfully!


In [5]:
data = open("dataset.txt", "r", encoding="utf-8").read()

dataset = Dataset.from_dict({"text": [data]})

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Dataset loaded successfully!")

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Dataset loaded successfully!


In [8]:
training_args = TrainingArguments(
    output_dir="./gpt2-output",
    num_train_epochs=10,
    per_device_train_batch_size=1,
    logging_steps=1,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
1,3.488374
2,2.909214
3,2.594167
4,2.270618
5,2.063260
6,1.913862
7,1.727510
8,1.680128
9,1.611337
10,1.474875


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=10, training_loss=2.1733346462249754, metrics={'train_runtime': 18.6591, 'train_samples_per_second': 0.536, 'train_steps_per_second': 0.536, 'total_flos': 653230080000.0, 'train_loss': 2.1733346462249754, 'epoch': 10.0})

In [11]:
import torch

prompt = "Artificial Intelligence"

inputs = tokenizer(prompt, return_tensors="pt")

# move inputs to same device as model
inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model.generate(
    **inputs,
    max_length=100,
    temperature=0.7,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Generated Text:\n")
print(generated_text)

Generated Text:

Artificial Intelligence is gaining in popularity across industries. In some industries, AI is being used in many industries, yet in some machines, it is not being used. However, AI is gaining in popularity in many industries.

With the growth in AI in IoT devices, it is becoming more common to automate the process. In this article, we will discuss the differences between automation and automation.

How automated products are being introduced into production systems

Automation in production products is being used
